# V7.5 — FRESH HARD-NEGATIVE / Bước 1: Tạo contact sheets để xác minh bằng mắt

**CPU ONLY.** 160 cặp ảnh–target mới trên FSC-147 validation, không trùng ảnh với 480 validation đã dùng cho V7.5 hoặc ảnh của bộ 111 hard-negative cũ.

**CỰC KỲ QUAN TRỌNG:** Các target chỉ là **ứng viên có vẻ gây nhầm lẫn**, không mặc định vắng mặt. Không lấy `target != source_class` làm bằng chứng đối tượng vắng mặt. Phải mở ảnh và kiểm tra bằng mắt.

Quy trình: chạy notebook, tải ZIP contact sheets; hai người kiểm tra độc lập, ghi vào CSV theo các giá trị `ABSENT`, `PRESENT`, `UNCERTAIN`; cột `adjudicated_presence=ABSENT` chỉ khi đã thống nhất không có đối tượng mục tiêu, sau đó mới chạy notebook suy luận bước 2.

In [ ]:
from pathlib import Path
import pandas as pd, zipfile, shutil, textwrap, math
from PIL import Image, ImageOps, ImageDraw, ImageFont
from IPython.display import FileLink, display
ROOT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT=ROOT/'v75_freshneg_review';OUT.mkdir(exist_ok=True)
INPUT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/mnt/data')
def get_csv(filename):
    found=sorted(INPUT.rglob(filename))
    if found:return pd.read_csv(found[0],keep_default_na=False)
    for f in sorted(INPUT.rglob('*.zip')):
        try:
            with zipfile.ZipFile(f) as z:
                names=[n for n in z.namelist() if Path(n).name==filename]
                if names:
                    import io
                    return pd.read_csv(io.BytesIO(z.read(names[0])),keep_default_na=False)
        except (zipfile.BadZipFile,PermissionError):continue
    raise FileNotFoundError('Add Input: V75_FreshNeg_Candidates_160.zip; missing '+filename)
rows=get_csv('V75_FreshNeg_Candidates_160.csv')
assert len(rows)==160 and rows.image_name.is_unique
images={p.name:p for p in INPUT.rglob('*.jpg')}
missing=sorted(set(rows.image_name)-set(images))
assert not missing, 'Missing FSC147 images: '+str(missing[:10])
print('Loaded',len(rows),'candidate image pairs; 0 labels assigned.')

In [ ]:
# Tạo 8 contact sheets, mỗi sheet 20 ảnh + source class + target
SHEETS=OUT/'contact_sheets';SHEETS.mkdir(exist_ok=True)
font_candidates=['/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf','/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf']
font=None
for f in font_candidates:
    if Path(f).exists():font=ImageFont.truetype(f,20);break
if font is None:font=ImageFont.load_default()
COLS=4; ROWS=5; CELL_W=390; CELL_H=300; IMAGE_H=218
for page,start in enumerate(range(0,len(rows),COLS*ROWS),1):
    part=rows.iloc[start:start+COLS*ROWS]
    canvas=Image.new('RGB',(COLS*CELL_W,ROWS*CELL_H),(247,247,247))
    d=ImageDraw.Draw(canvas)
    for idx,(_,r) in enumerate(part.iterrows()):
        x=(idx%COLS)*CELL_W;y=(idx//COLS)*CELL_H
        with Image.open(images[r.image_name]) as im:
            im=im.convert('RGB');im.thumbnail((CELL_W-14,IMAGE_H-8))
            canvas.paste(im,(x+(CELL_W-im.width)//2,y+(IMAGE_H-im.height)//2))
        d.rectangle((x+1,y+1,x+CELL_W-2,y+CELL_H-2),outline=(170,170,170),width=2)
        labels=[f'{r.candidate_id} | {r.image_name} | {r.candidate_type}',
                f'SOURCE: {r.source_class}',f'QUERY: count {r.target}']
        for k,line in enumerate(labels):
            d.text((x+8,y+IMAGE_H+5+k*24),line,font=font,fill=(0,0,0))
    path=SHEETS/f'freshneg_contact_sheet_{page:02}.jpg'
    canvas.save(path,quality=90)
    print('Saved',path.name)
rows.to_csv(OUT/'V75_FreshNeg_Reviewed_160.csv',index=False,encoding='utf-8-sig')
zipout=ROOT/'V75_FreshNeg_Review_Pack.zip'
with zipfile.ZipFile(zipout,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(OUT/'V75_FreshNeg_Reviewed_160.csv','V75_FreshNeg_Reviewed_160.csv')
    for f in sorted(SHEETS.glob('*.jpg')):z.write(f,'contact_sheets/'+f.name)
print('DONE:',zipout)
display(FileLink(str(zipout)))

### Cách điền bảng sau khi xem contact sheets

- `reviewer_1_decision`: `ABSENT`, `PRESENT` hoặc `UNCERTAIN`.
- `reviewer_2_decision`: người thứ hai đánh giá độc lập với bộ giá trị tương tự (rất nên có khi báo cáo paper).
- `adjudicated_presence`: `ABSENT` **chỉ khi người kiểm tra kết luận đối tượng không xuất hiện trong ảnh**. Mâu thuẫn hoặc hình quá mờ ⇒ `UNCERTAIN`.
- `review_notes`: ghi lý do không chắc chắn/đối tượng gây nhầm lẫn nếu có.

**Không tự động điền `ABSENT` cho cả 160 dòng.** Lưu bảng đã duyệt với tên chính xác `V75_FreshNeg_Reviewed_160.csv` và Add Input vào Kaggle ở notebook bước 2. Với kết quả paper cần hai người đánh giá độc lập và nhất trí, hoặc quy trình phân xử có ghi nhận.